# Ingesta bronze - aisles y departments
Son dos tablas pequenas de referencia, por lo que se cargan con una misma funcion que recibe la entidad y su schema.

In [ ]:
%run ./00_utils

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

params = obtener_parametros()
CATALOGO, STORAGE = params["catalogo"], params["storage"]

ENTIDADES = {
    "aisles": StructType([
        StructField("aisle_id", IntegerType(), False),
        StructField("aisle", StringType(), True),
    ]),
    "departments": StructType([
        StructField("department_id", IntegerType(), False),
        StructField("department", StringType(), True),
    ]),
}

In [ ]:
def ingestar(entidad, schema):
    ruta = ruta_raw(STORAGE, entidad)
    log(f"Leyendo {ruta}", entidad=entidad)
    df_raw = (spark.read
              .option("header", True)
              .option("sep", ";")
              .schema(schema)
              .csv(ruta))
    df_bronze = agregar_auditoria(df_raw)
    validar_no_vacio(df_bronze, entidad)
    escribir_tabla(df_bronze, f"{CATALOGO}.bronze.{entidad}", modo="overwrite")

In [ ]:
for entidad, schema in ENTIDADES.items():
    try:
        ingestar(entidad, schema)
    except Exception as e:
        log(f"Error en la ingesta: {e}", "ERROR", entidad)
        raise

In [ ]:
display(spark.table(f"{CATALOGO}.bronze.departments"))